In [1]:
import os
import pandas as pd
import psycopg2
from psycopg2.extras import RealDictCursor

## Konfiguration

In [2]:
# --- Standort ---
lat = 49.3647
lon = 8.1266

# --- Zeitraum ---
start_time = '2023-07-24'
end_time   = '2026-02-28'

# --- Filter (None = kein Filter) ---
forecasttime_min = 1       # None = alle
forecasttime_max = 48      # None = alle
toplevel_filter  = None    # z.B. 1000.0 für einen bestimmten Drucklevel, None = alle

# --- Spalten (None = alle) ---
# z.B. ['starttime', 'forecasttime', 'toplevel', 'bottomlevel', 'u_wind', 'v_wind', 'temperature', 'pressure', 'qs']
select_cols = None

## Datenbankverbindung

In [3]:
db_url = os.environ.get('WEATHER_DB_URL')
if not db_url:
    raise EnvironmentError("WEATHER_DB_URL nicht gesetzt – bitte in .bashrc eintragen.")

conn = psycopg2.connect(db_url)
print("Verbindung OK:", conn.get_dsn_parameters()['dbname'])

Verbindung OK: WeatherDB


## Schritt 1 — Nächsten Gitterpunkt via `icon_d2_grid_points` (KNN)

**Warum:** `icon_d2_grid_points` ist eine kleine Hilfstabelle mit nur den Grid-Geometrien.  
Der `<->` KNN-Operator nutzt den GiST-Index direkt → viel schneller als `ST_DWithin` auf `multilevelfields`.

> **Achtung:** `icon_d2_grid_points` speichert `POINT(lat, lon)` (nicht-standard), d.h. `ST_X = lat`, `ST_Y = lon`.

In [5]:
# icon_d2_grid_points: POINT(lat, lon) → ST_X = lat, ST_Y = lon
knn_sql = """
    SELECT
        ST_X(geom) AS grid_lat,
        ST_Y(geom) AS grid_lon
    FROM icon_d2_grid_points
    ORDER BY geom <-> ST_SetSRID(ST_MakePoint(%(lat)s, %(lon)s), 4326)
    LIMIT 1;
"""

with conn.cursor(cursor_factory=RealDictCursor) as cur:
    cur.execute(knn_sql, {'lat': lat, 'lon': lon})
    gp = cur.fetchone()

if gp is None:
    raise ValueError(f"Kein Gitterpunkt in icon_d2_grid_points gefunden.")

grid_lat = float(gp['grid_lat'])
grid_lon = float(gp['grid_lon'])
print(f"Gesuchter Punkt  : lat={lat}, lon={lon}")
print(f"Nächster Gitterpunkt: lat={grid_lat:.4f}, lon={grid_lon:.4f}")

Gesuchter Punkt  : lat=49.3647, lon=8.1266
Nächster Gitterpunkt: lat=13.3825, lon=48.5293


## Schritt 2 — Exakte `geom` aus `multilevelfields` auflösen

Wir brauchen die binäre Geom aus `multilevelfields` (standard `POINT(lon, lat)`),  
damit der Haupt-Query `WHERE geom = %s` den primären Index nutzt.

In [6]:
# multilevelfields: POINT(lon, lat) → ST_MakePoint(grid_lon, grid_lat)
resolve_geom_sql = """
    SELECT geom
    FROM multilevelfields
    WHERE ST_DWithin(
        geom,
        ST_SetSRID(ST_MakePoint(%(grid_lon)s, %(grid_lat)s), 4326),
        0.001
    )
    LIMIT 1;
"""

with conn.cursor() as cur:
    cur.execute(resolve_geom_sql, {'grid_lon': grid_lon, 'grid_lat': grid_lat})
    row = cur.fetchone()

if row is None:
    raise ValueError(
        f"Kein Eintrag in multilevelfields für ({grid_lat:.4f}, {grid_lon:.4f}). "
        f"Gitterpunkt existiert in icon_d2_grid_points aber nicht in multilevelfields?"
    )

exact_geom = row[0]
print(f"Exakte geom aufgelöst (geom-bytes Länge: {len(exact_geom)})")

Exakte geom aufgelöst (geom-bytes Länge: 50)


## Schritt 3 — Daten abfragen (`WHERE geom = %s`)

Direkter Index-Hit auf den primären GiST/B-Tree-Index von `multilevelfields`.

In [7]:
col_expr = '*' if not select_cols else ', '.join(select_cols)

conditions = [
    "geom = %(geom)s",
    "starttime >= %(start_time)s",
    "starttime < %(end_time)s",
]
params = dict(geom=exact_geom, start_time=start_time, end_time=end_time)

if forecasttime_min is not None:
    conditions.append("forecasttime >= %(ft_min)s")
    params['ft_min'] = forecasttime_min
if forecasttime_max is not None:
    conditions.append("forecasttime <= %(ft_max)s")
    params['ft_max'] = forecasttime_max
if toplevel_filter is not None:
    conditions.append("ABS(toplevel - %(toplevel)s) < 0.01")
    params['toplevel'] = toplevel_filter

query = f"""
    SELECT {col_expr}
    FROM multilevelfields
    WHERE {' AND '.join(conditions)}
    ORDER BY starttime, forecasttime, toplevel;
"""

with conn.cursor() as cur:
    cur.execute(query, params)
    rows    = cur.fetchall()
    columns = [desc.name for desc in cur.description]

df = pd.DataFrame(rows, columns=columns)
print(f"{len(df):,} Zeilen geladen  |  Spalten: {list(df.columns)}")
df.head()

24,078 Zeilen geladen  |  Spalten: ['starttime', 'forecasttime', 'geom', 'toplevel', 'bottomlevel', 'u_wind', 'v_wind', 'temperature', 'pressure', 'qs', 'relhum']


,starttime,forecasttime,geom,toplevel,bottomlevel,u_wind,v_wind,temperature,pressure,qs,relhum
0,2023-07-24 08:00:00+02:00,1.0,0101000020E6100000FC19DEACC1434840A1A014ADDCC3...,20.000,0.000,-0.786334,2.359617,293.64000,96033.984,0.009862,None
1,2023-07-24 08:00:00+02:00,1.0,0101000020E6100000FC19DEACC1434840A1A014ADDCC3...,55.212,20.000,-0.974445,2.737845,293.23975,95742.336,0.009754,None
2,2023-07-24 08:00:00+02:00,1.0,0101000020E6100000FC19DEACC1434840A1A014ADDCC3...,100.277,55.212,-1.107566,2.910645,292.84583,95327.016,0.009696,None
3,2023-07-24 08:00:00+02:00,1.0,0101000020E6100000FC19DEACC1434840A1A014ADDCC3...,153.438,100.277,-2.382144,4.167841,293.18230,94819.016,0.008813,None
4,2023-07-24 08:00:00+02:00,1.0,0101000020E6100000FC19DEACC1434840A1A014ADDCC3...,213.746,153.438,-3.768396,5.267670,293.92126,94239.940,0.008463,None


In [10]:
df

,starttime,forecasttime,geom,toplevel,bottomlevel,u_wind,v_wind,temperature,pressure,qs,relhum
0,2023-07-24 08:00:00+02:00,1.0,0101000020E6100000FC19DEACC1434840A1A014ADDCC3...,20.000,0.000,-0.786334,2.359617,293.64000,96033.984,0.009862,None
1,2023-07-24 08:00:00+02:00,1.0,0101000020E6100000FC19DEACC1434840A1A014ADDCC3...,55.212,20.000,-0.974445,2.737845,293.23975,95742.336,0.009754,None
2,2023-07-24 08:00:00+02:00,1.0,0101000020E6100000FC19DEACC1434840A1A014ADDCC3...,100.277,55.212,-1.107566,2.910645,292.84583,95327.016,0.009696,None
3,2023-07-24 08:00:00+02:00,1.0,0101000020E6100000FC19DEACC1434840A1A014ADDCC3...,153.438,100.277,-2.382144,4.167841,293.18230,94819.016,0.008813,None
4,2023-07-24 08:00:00+02:00,1.0,0101000020E6100000FC19DEACC1434840A1A014ADDCC3...,213.746,153.438,-3.768396,5.267670,293.92126,94239.940,0.008463,None
...,...,...,...,...,...,...,...,...,...,...,...
24073,2023-10-15 08:00:00+02:00,29.0,0101000020E6100000FC19DEACC1434840A1A014ADDCC3...,55.212,20.000,-0.433058,-1.302447,280.85776,96831.695,0.003915,None
24074,2023-10-15 08:00:00+02:00,29.0,0101000020E6100000FC19DEACC1434840A1A014ADDCC3...,100.277,55.212,-0.471083,-1.331238,280.39822,96391.390,0.003888,None
24075,2023-10-15 08:00:00+02:00,29.0,0101000020E6100000FC19DEACC1434840A1A014ADDCC3...,153.438,100.277,-0.502159,-1.330347,279.89380,95853.350,0.003871,None
24076,2023-10-15 08:00:00+02:00,29.0,0101000020E6100000FC19DEACC1434840A1A014ADDCC3...,213.746,153.438,-0.531050,-1.313707,279.33936,95237.190,0.003857,None


## Schnell-Check

In [8]:
nan_summary = df.isna().sum()
print("NaN pro Spalte:")
display(nan_summary[nan_summary > 0] if nan_summary.any() else "Keine NaN-Werte.")

NaN pro Spalte:


relhum    24078
dtype: int64

In [9]:
# Verfügbare toplevel-Werte und forecasttime-Abdeckung
if 'toplevel' in df.columns:
    print("toplevel-Werte:", sorted(df['toplevel'].unique()))
if 'forecasttime' in df.columns:
    ft = sorted(df['forecasttime'].unique())
    print(f"forecasttime: {ft[0]} … {ft[-1]}  ({len(ft)} Werte)")
if 'starttime' in df.columns:
    st = df['starttime'].nunique()
    print(f"Anzahl Runs (starttime): {st}")

toplevel-Werte: [20.0, 55.212, 100.277, 153.438, 213.746, 280.598]
forecasttime: 1.0 … 48.0  (48 Werte)
Anzahl Runs (starttime): 84


In [ ]:
conn.close()